# main

In [ ]:
#|default_exp main
#|export_as_func true

In [ ]:
#|hide
import nblite; from nbdev.showdoc import show_doc; nblite.nbl_export()

In [ ]:
#|top_export
import typer
from typer import Argument, Option
from typing_extensions import Annotated
from pathlib import Path
import toml
import json

from add_to_pet import const
from add_to_pet.app import app

In [ ]:
#|set_func_signature
@app.command()
def add_to_pet(
    cmd: Annotated[str, Argument(help="The command to add to pet.")],
    description: Annotated[str, Option("-d", "--description", help="The description of the command.")] = "",
    output: Annotated[str, Option("-o", "--output", help="The output of the command.")] = "",
    tag: Annotated[list[str], Option("-t", "--tag", help="The tags of the command.")] = [],
    name: Annotated[str, Option("-n", "--name", help="The name of the command. Will be added to ~/.config/pet/aliases.sh, as well as to the description.")] = None,
    interactive: Annotated[bool, Option("-i", "--interactive", help="Whether to run the command interactively.")] = False,
    snippets_path: Annotated[str|None, Option("--snippets-path", help="Path to the snippets file.")] = None,
    aliases_path: Annotated[str|None, Option("--aliases-path", help="Path to the aliases file.")] = None,
): ...

In [ ]:
cmd = "ls -l"
description = "List the contents of the current directory in long format"
output = ""
tag = ["list", "directory"]
name = "lll"
interactive = False
snippets_path = "./test_snippet.toml"
aliases_path = "./test_aliases.sh"

In [ ]:
#|export
if interactive:
    name = input("Name: ") if name is None else name
    description = input("Description: ") if not description else description
    tag = [t.strip() for t in input("Tags (comma separated): ").split(",")] if not tag else tag

In [ ]:
#|export
if name:
    _description = f"({name}) {description}"
else:
    _description = description

snippet = {
    "command": cmd,
    "Description": _description,
    "Output": output,
    "Tag": tag,
    "name": name,
}

In [ ]:
#|export
snippets_path = const.snippets_path if snippets_path is None else Path(snippets_path)
snippets_data = toml.loads(open(snippets_path).read())
if 'Snippets' not in snippets_data:
    snippets_data['Snippets'] = []
duplicate_snippets = [i for i, s in enumerate(snippets_data["Snippets"]) if s["command"] == cmd]

if len(duplicate_snippets) == 1:
    snippet_index = duplicate_snippets[0]
    snippets_data["Snippets"][snippet_index] = snippet
elif len(duplicate_snippets) > 1:
    raise ValueError(f"Found {len(duplicate_snippets)} duplicate snippets for command {cmd}. Maximum number of duplicates allowed is 1.")
else:
    snippets_data["Snippets"].append(snippet)

with open(snippets_path, "w") as f:
    f.write(toml.dumps(snippets_data))

Create the `alias.sh` file

In [ ]:
#|export
import shlex

aliases_path = const.aliases_path if aliases_path is None else Path(aliases_path)
if not aliases_path.exists():
    aliases_path.touch()

aliases = []

for snippet in snippets_data["Snippets"]:
    if 'name' not in snippet: continue
    aliases.append(f"alias {snippet['name']}={shlex.quote(snippet['command'])}")

aliases_path.write_text("\n".join(aliases));